# Stage 1 — Qwen3-1.7B Text-to-SQL Baseline (Spider)

Orchestrator notebook only — all real logic lives in `src/*.py` in the project folder. This notebook just: mounts Drive, installs deps, downloads the Spider dataset + eval suite, then shells out to the existing scripts.

**One-time manual prep before running this notebook:**
1. Upload this whole `finetuning project` folder to your Google Drive (e.g. `MyDrive/finetuning_project/`).

(Spider itself now downloads automatically — see the cell below.)

Runtime: make sure **Runtime > Change runtime type > T4 GPU** is selected before running.

In [1]:
!nvidia-smi

Mon Jul  6 08:45:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Change this if you uploaded the project to a different Drive path.
PROJECT_DIR = '/content/drive/MyDrive/finetuning_project'

required = ['src/generate_predictions.py', 'src/run_eval.py', 'requirements.txt']
missing = [f for f in required if not os.path.exists(os.path.join(PROJECT_DIR, f))]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under {PROJECT_DIR}. "
        "Upload the finetuning project folder to this Drive path first."
    )
print('Project files found at', PROJECT_DIR)

Project files found at /content/drive/MyDrive/finetuning_project


In [4]:
%cd {PROJECT_DIR}

/content/drive/MyDrive/finetuning_project


In [ ]:
!pip install -q -r requirements.txt

## Download + unzip Spider

Downloads `spider_data.zip` directly from the official Google Drive link on https://yale-lily.github.io/spider (verified file ID) using `gdown`, which handles Google Drive's large-file confirmation flow that plain `curl`/`wget` can't. Safe to re-run — skips if already downloaded/extracted.

In [5]:
!pip install -q gdown

import os
import zipfile, shutil, glob

SPIDER_GDRIVE_ID = '1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J'  # verified against yale-lily.github.io/spider
SPIDER_ZIP_PATH = f'{PROJECT_DIR}/spider_data.zip'
target_dir = 'data/spider'

if os.path.exists(os.path.join(target_dir, 'tables.json')):
    print('Spider already extracted at', target_dir)
else:
    if not os.path.exists(SPIDER_ZIP_PATH):
        print('Downloading spider_data.zip (~1GB, can take a few minutes)...')
        import gdown
        try:
            gdown.download(id=SPIDER_GDRIVE_ID, output=SPIDER_ZIP_PATH, quiet=False)
        except Exception as e:
            if os.path.exists(SPIDER_ZIP_PATH):
                os.remove(SPIDER_ZIP_PATH)  # don't leave a partial/HTML file behind
            raise RuntimeError(
                "gdown download failed -- Spider is a heavily-downloaded dataset and Google "
                "Drive sometimes rate-limits it ('too many users have viewed/downloaded this "
                "file'). Fix: download it manually via browser from "
                "https://yale-lily.github.io/spider, upload the zip to "
                f"{SPIDER_ZIP_PATH}, then re-run this cell."
            ) from e
        if not os.path.exists(SPIDER_ZIP_PATH) or os.path.getsize(SPIDER_ZIP_PATH) < 10_000_000:
            raise RuntimeError(
                f"Download at {SPIDER_ZIP_PATH} is missing or suspiciously small -- likely an "
                "HTML rate-limit page saved as a zip, not the real dataset. Delete it and either "
                "retry this cell later, or download manually via browser and upload it here."
            )
    else:
        print('Zip already present at', SPIDER_ZIP_PATH)

    extract_tmp = 'data/_spider_extract_tmp'
    os.makedirs(extract_tmp, exist_ok=True)
    print('Unzipping...')
    with zipfile.ZipFile(SPIDER_ZIP_PATH) as zf:
        zf.extractall(extract_tmp)

    # The zip nests everything under a subfolder -- find wherever tables.json landed.
    matches = glob.glob(os.path.join(extract_tmp, '**', 'tables.json'), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Couldn't find tables.json anywhere inside {SPIDER_ZIP_PATH}. "
                                 "The zip may be corrupted -- delete it and re-run this cell.")
    spider_root = os.path.dirname(matches[0])

    os.makedirs('data', exist_ok=True)
    shutil.move(spider_root, target_dir)
    shutil.rmtree(extract_tmp, ignore_errors=True)
    print('Spider extracted to', target_dir)

Downloading...
From (original): https://drive.google.com/uc?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J
From (redirected): https://drive.google.com/uc?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J&confirm=t&uuid=4a3da67b-4612-42fd-970d-792e143fc6ea
To: /content/drive/MyDrive/finetuning_project/spider_data.zip
100%|██████████| 206M/206M [00:02<00:00, 70.9MB/s]


Unzipping...
Spider extracted to data/spider


In [7]:
# Sanity-check Spider data is actually in place before wasting a generation run on it.
spider_required = ['data/spider/dev.json', 'data/spider/tables.json', 'data/spider/database']
missing_data = [f for f in spider_required if not os.path.exists(f)]
if missing_data:
    raise FileNotFoundError(
        f"Missing {missing_data}. Download Spider from https://yale-lily.github.io/spider, "
        f"unzip it, and place the contents under {PROJECT_DIR}/data/spider/"
    )
print('Spider data found.')

Spider data found.


In [8]:
# Official eval suite -- public repo, safe to auto-clone.
import os
eval_suite_dir = 'third_party/test_suite_sql_eval'
if not os.path.exists(os.path.join(eval_suite_dir, 'evaluation.py')):
    !git clone https://github.com/taoyds/test-suite-sql-eval {eval_suite_dir}
else:
    print('Eval suite already present.')

Cloning into 'third_party/test_suite_sql_eval'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 61 (delta 20), reused 16 (delta 16), pack-reused 31 (from 2)
Receiving objects: 100% (61/61), 619.62 KiB | 4.84 MiB/s, done.
Resolving deltas: 100% (24/24), done.


## Smoke test (10 examples)

Run this first -- cheap sanity check that generation works on this runtime before committing to the full ~1000-example dev set.

Note: T4 is Turing architecture and doesn't have native bf16 tensor-core support, so we pass `--dtype float16` here instead of the script's default bf16.

In [9]:
!python src/generate_predictions.py \
  --model_name Qwen/Qwen3-1.7B \
  --data_dir data/spider \
  --output_dir results/baseline_qwen3_1p7b_smoke \
  --dtype float16 \
  --limit 10

config.json: 100% 726/726 [00:00<00:00, 2.79MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 15.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 90.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 123MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:00<00:00, 13.1MB/s]
model.safetensors.index.json: 100% 25.6k/25.6k [00:00<00:00, 76.5MB/s]
Fetching 2 files: 100% 2/2 [00:29<00:00, 14.81s/it]
Download complete: 100% 4.06G/4.06G [00:30<00:00, 135MB/s]
Loading weights: 100% 311/311 [00:14<00:00, 21.39it/s]
generation_config.json: 100% 239/239 [00:00<00:00, 939kB/s]
Generating: 100% 10/10 [00:23<00:00,  2.34s/it]
{
  "model_name": "Qwen/Qwen3-1.7B",
  "num_examples": 10,
  "unsafe_predictions": 0,
  "total_generation_time_sec": 23.316447435000214,
  "avg_latency_sec": 2.3316447435000214,
  "tokens_per_sec": 18.098812058587352,
  "peak_gpu_memory_gb": 3.2687010765075684
}


In [12]:
!python src/run_eval.py \
  --gold results/baseline_qwen3_1p7b_smoke/gold.txt \
  --pred results/baseline_qwen3_1p7b_smoke/pred.txt \
  --db data/spider/database \
  --table data/spider/tables.json \
  --output_dir results/baseline_qwen3_1p7b_smoke

medium pred: SELECT s.Name, s.Country, s.Age FROM singer s ORDER BY s.Age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT s.Name, s.Country, s.Age FROM singer s JOIN singer_in_concert sic ON s.Singer_ID = sic.Singer_ID JOIN concert c ON sic.concert_ID = c.concert_ID JOIN stadium st ON c.Stadium_ID = st.Stadium_ID ORDER BY s.Age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT AVG(Age) AS Average, MIN(Age) AS Minimum, MAX(Age) AS Maximum FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

medium pred: SELECT AVG(Age) AS Average, MIN(Age) AS Minimum, MAX(Age) AS Maximum FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

medium pred: SELECT s.Name, sc.Song_release_year FROM singer s JOIN singer_in_concert sic ON s.Singer_ID = sic.Singe

## Full baseline run (full dev set)

Only run this once the smoke test above looks sane.

In [13]:
!python src/generate_predictions.py \
  --model_name Qwen/Qwen3-1.7B \
  --data_dir data/spider \
  --output_dir results/baseline_qwen3_1p7b \
  --dtype float16

Loading weights: 100% 311/311 [00:05<00:00, 51.92it/s]
Generating: 100% 1034/1034 [30:04<00:00,  1.75s/it]
{
  "model_name": "Qwen/Qwen3-1.7B",
  "num_examples": 1034,
  "unsafe_predictions": 1,
  "total_generation_time_sec": 1799.6665099190036,
  "avg_latency_sec": 1.7404898548539687,
  "tokens_per_sec": 19.305243392879305,
  "peak_gpu_memory_gb": 3.393155097961426
}


In [14]:
!python src/run_eval.py \
  --gold results/baseline_qwen3_1p7b/gold.txt \
  --pred results/baseline_qwen3_1p7b/pred.txt \
  --db data/spider/database \
  --table data/spider/tables.json \
  --output_dir results/baseline_qwen3_1p7b

medium pred: SELECT s.Name, s.Country, s.Age FROM singer s ORDER BY s.Age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT s.Name, s.Country, s.Age FROM singer s JOIN singer_in_concert sic ON s.Singer_ID = sic.Singer_ID JOIN concert c ON sic.concert_ID = c.concert_ID JOIN stadium st ON c.Stadium_ID = st.Stadium_ID ORDER BY s.Age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT AVG(Age) AS Average, MIN(Age) AS Minimum, MAX(Age) AS Maximum FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

medium pred: SELECT AVG(Age) AS Average, MIN(Age) AS Minimum, MAX(Age) AS Maximum FROM singer WHERE Country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

medium pred: SELECT s.Name, sc.Song_release_year FROM singer s JOIN singer_in_concert sic ON s.Singer_ID = sic.Singe

## Results

Check, in `results/baseline_qwen3_1p7b/`:
- `eval_log.txt` -- Exact Match + Execution Accuracy, broken down by difficulty
- `perf_metrics.json` -- avg latency, tokens/sec, peak GPU memory
- `predictions_debug.jsonl` -- per-example question/gold/prediction/latency, for error analysis

Because everything is stored on Drive (`PROJECT_DIR`), these results survive after the Colab runtime disconnects.

**Next stage**: error analysis on `predictions_debug.jsonl`, then fine-tuning with Unsloth -- not yet scaffolded, comes after baseline numbers are reviewed.